# Consumer360 — Week 1 Data Foundation Audit

A read-only audit of the Week 1 deliverables: source data reconciliation, PostgreSQL staging, data quality signals, transaction classification, star schema, declared referential integrity, Single Customer View, repository SQL assets, and historical plus optimized SQL performance evidence.

The notebook does not modify source files or database objects. It uses the project's existing local configuration where available. Missing prerequisites are reported in the audit results; they do not require manual notebook edits.

The optimized benchmark values recorded in this notebook are the latest measured local results supplied by the project audit. They are clearly distinguished from timings freshly measured during a notebook execution.


## 1. Scope and reported baseline

Reference values from `Consumer360_Week1_Completion_and_Details.docx`:

| Measure | Reported value |
|---|---:|
| 2009–10 source rows | 525,461 |
| 2010–11 source rows | 541,910 |
| Cross-file matching occurrences | 22,523 |
| Staging / fact rows | 1,044,848 |
| `dim_customer` rows including unknown | 5,943 |
| `dim_product` rows including unknown | 5,305 |
| Identified customers in Single Customer View | 5,942 |
| Merchandise purchasers | 5,852 |
| Customers without merchandise orders | 90 |
| SCV merchandise revenue | £17,117,089.2970 |
| SQL performance target | under 2,000 ms |

The report says overlap was excluded from the second source file only; within-file duplicates were preserved. Source files were not modified.

In [1]:
from pathlib import Path
import hashlib, sys
import pandas as pd
from IPython.display import display

# Find the project root from the notebook's current working directory or its parents.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "pyproject.toml").exists() and (p / "database").exists()), Path.cwd())
RAW = ROOT / "data" / "raw"
REPORT = ROOT / "reports" / "ingestion_report.json"
MIGRATIONS = ROOT / "database" / "migrations"
ERD = ROOT / "database" / "diagrams" / "consumer360_star_schema.mmd"
BENCHMARK = ROOT / "database" / "queries" / "006_performance_benchmark.sql"
FILES = [RAW / "online_retail_II_2009-10.csv", RAW / "online_retail_II_2010-11.csv"]
DATE_FORMAT = "%d-%m-%y %H:%M"
COLUMNS = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]

BASELINE = {"source_2009":525461, "source_2010":541910, "overlap":22523,
            "staging":1044848, "customers":5943, "products":5305, "fact":1044848,
            "scv_customers":5942, "purchasers":5852, "no_orders":90,
            "scv_revenue":17117089.2970, "target_ms":2000.0}
checks = []

def add_check(name, status, observed=None, expected=None, note=None):
    checks.append({"check":name,"status":status,"observed":observed,"expected":expected,"note":note})

def compare(name, observed, expected, tol=0):
    if observed is None or pd.isna(observed):
        status = "NOT VERIFIED"
    elif isinstance(expected, (int,float)):
        status = "PASS" if abs(float(observed)-float(expected)) <= tol else "REVIEW"
    else:
        status = "PASS" if observed == expected else "REVIEW"
    add_check(name,status,observed,expected)

print("Project root:", ROOT)
print("Python:", sys.version.split()[0])
from sqlalchemy import text


Project root: c:\Users\acer\Downloads\Consumer-360
Python: 3.11.17


## 2. Source data audit

The eight expected fields are `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, and `Country`. Identifiers are loaded as text so numeric coercion cannot alter them.

In [2]:
frames, source_rows = {}, []
for path in FILES:
    if not path.exists():
        add_check(f"Source file: {path.name}", "NOT VERIFIED", "Not found", "Available in data/raw")
        continue
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", dtype="string", low_memory=False)
        frames[path.name] = df
        missing = sorted(set(COLUMNS) - set(df.columns))
        if not missing:
            dates = pd.to_datetime(df["InvoiceDate"], format=DATE_FORMAT, errors="coerce")
            min_date, max_date = dates.min(), dates.max()
        else:
            min_date = max_date = None
        source_rows.append({"file":path.name,"rows":len(df),"columns":len(df.columns),
            "missing_required_columns":missing,"min_date":min_date,"max_date":max_date,
            "size_bytes":path.stat().st_size,
            "sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
        add_check(f"Required columns: {path.name}", "PASS" if not missing else "FAIL", missing, [])
    except Exception as exc:
        add_check(f"Read source file: {path.name}", "NOT VERIFIED", type(exc).__name__, "Readable CSV")

source_audit = pd.DataFrame(source_rows)
display(source_audit if not source_audit.empty else pd.DataFrame(columns=["file","rows","columns","missing_required_columns","min_date","max_date","size_bytes","sha256"]))
if len(source_rows) == 2:
    rows_by_file = {r["file"]: r["rows"] for r in source_rows}
    compare("2009–10 source row count", rows_by_file.get(FILES[0].name), BASELINE["source_2009"])
    compare("2010–11 source row count", rows_by_file.get(FILES[1].name), BASELINE["source_2010"])

if len(frames) == 2 and all(set(COLUMNS).issubset(frames[p.name].columns) for p in FILES):
    left, right = [frames[p.name][COLUMNS].fillna("<NULL>").astype(str) for p in FILES]
    lc, rc = left.value_counts(), right.value_counts()
    overlap = int(pd.concat([lc,rc],axis=1).fillna(0).min(axis=1).sum())
    retained = len(left) + len(right) - overlap
    print("Cross-file matching occurrences:", overlap)
    compare("Cross-file overlap occurrences", overlap, BASELINE["overlap"])
    compare("Retained rows after overlap handling", retained, BASELINE["staging"])
else:
    add_check("Cross-file overlap", "NOT VERIFIED", "Both readable files with required columns are needed", BASELINE["overlap"])


,file,rows,columns,missing_required_columns,min_date,max_date,size_bytes,sha256
0,online_retail_II_2009-10.csv,525461,8,[],2009-12-01 07:45:00,2010-12-09 20:01:00,43820738,1304229e8dd9e8513bbb7a3ee2bf98baf7374034adaa27...
1,online_retail_II_2010-11.csv,541910,8,[],2010-12-01 08:26:00,2011-12-09 12:50:00,45005814,d568eddb5e9b906a08b2f07690909c8322d5d269a02a6a...


Cross-file matching occurrences: 22523


## 3. PostgreSQL connection and staging audit

The project's existing connection helper loads credentials from the local `.env` file. Credentials are never displayed. Database queries are read-only. If PostgreSQL is unavailable, database checks are marked `NOT VERIFIED` and the notebook continues.


In [3]:
engine = None
try:
    sys.path.insert(0, str(ROOT / "src"))
    from consumer360.database.connection import get_engine
    engine = get_engine()
    with engine.connect() as conn:
        identity = pd.read_sql_query(text("SELECT current_database() AS database_name, current_user AS database_user, version() AS server_version"), conn)
    display(identity)
    add_check("PostgreSQL connection","PASS","Connected","Connected")
except Exception as exc:
    print("Connection unavailable:",type(exc).__name__,str(exc)[:250])
    add_check("PostgreSQL connection","NOT VERIFIED",type(exc).__name__,"Connected")

if engine is not None:
    try:
        with engine.connect() as conn:
            staging = pd.read_sql_query(text("""
                SELECT COUNT(*) AS row_count,
                       COUNT(DISTINCT staging_id) AS distinct_staging_ids,
                       COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_id,
                       COUNT(*) FILTER (WHERE description IS NULL) AS missing_description,
                       COUNT(*) FILTER (WHERE unit_price = 0) AS zero_unit_price,
                       COUNT(*) FILTER (WHERE unit_price < 0) AS negative_unit_price,
                       COUNT(*) FILTER (WHERE quantity < 0) AS negative_quantity,
                       COUNT(*) FILTER (WHERE is_cancellation) AS cancellation_rows,
                       COUNT(*) FILTER (WHERE source_file IS NULL OR source_row_number IS NULL) AS missing_lineage,
                       MIN(invoice_date) AS first_invoice_date,
                       MAX(invoice_date) AS last_invoice_date
                FROM staging.retail_transactions
            """), conn)
        display(staging)
        s = staging.iloc[0]
        compare("Staging row count",int(s.row_count),BASELINE["staging"])
        compare("Staging ID uniqueness",int(s.distinct_staging_ids),int(s.row_count))
        compare("Staging lineage completeness",int(s.missing_lineage),0)
    except Exception as exc:
        print("Staging query unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Staging audit","NOT VERIFIED",type(exc).__name__,"staging.retail_transactions query")

,database_name,database_user,server_version
0,consumer360,postgres,"PostgreSQL 18.6 on x86_64-windows, compiled by..."


,row_count,distinct_staging_ids,missing_customer_id,missing_description,zero_unit_price,negative_unit_price,negative_quantity,cancellation_rows,missing_lineage,first_invoice_date,last_invoice_date
0,1044848,1044848,235287,4275,6024,5,22557,19165,0,2009-12-01 07:45:00,2011-12-09 12:50:00


## 4. Data quality and classification audit

Cancellation invoice flags and negative quantities are separate signals. The Week 1 report records 19,165 cancellation-flagged rows, 22,557 negative-quantity rows, 19,164 with both, 3,393 negative quantities without cancellation flags, and one cancellation flag without negative quantity.

The v2 classification is additive and retains all 1,044,848 source lines.

In [4]:
if engine is not None:
    try:
        with engine.connect() as conn:
            signals = pd.read_sql_query(text("""
                SELECT COUNT(*) FILTER (WHERE is_cancellation) AS cancellation_flagged,
                       COUNT(*) FILTER (WHERE quantity < 0) AS negative_quantity,
                       COUNT(*) FILTER (WHERE is_cancellation AND quantity < 0) AS both_signals,
                       COUNT(*) FILTER (WHERE NOT is_cancellation AND quantity < 0) AS negative_qty_without_cancellation,
                       COUNT(*) FILTER (WHERE is_cancellation AND quantity >= 0) AS cancellation_without_negative_qty
                FROM staging.retail_transactions
            """), conn)
            categories = pd.read_sql_query(text("""
                SELECT source_category, reporting_category, COUNT(*) AS row_count
                FROM analytics.fact_sales
                GROUP BY source_category, reporting_category
                ORDER BY source_category, reporting_category
            """), conn)
        display(signals); display(categories)
        compare("Classification row preservation",int(categories.row_count.sum()),BASELINE["fact"])
    except Exception as exc:
        print("Classification audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Classification audit","NOT VERIFIED",type(exc).__name__,"Staging signals and fact categories")

,cancellation_flagged,negative_quantity,both_signals,negative_qty_without_cancellation,cancellation_without_negative_qty
0,19165,22557,19164,3393,1


,source_category,reporting_category,row_count
0,BAD_DEBT_ADJUSTMENT,BAD_DEBT_ADJUSTMENT,6
1,BANK_CHARGE,BANK_CHARGE,100
2,CANCELLATION,CANCELLATION,18035
3,DISCOUNT_ADJUSTMENT,DISCOUNT_ADJUSTMENT,173
4,MANUAL_ADJUSTMENT,MANUAL_ADJUSTMENT,1473
5,OPERATIONAL_ADJUSTMENT,OPERATIONAL_ADJUSTMENT,842
6,POSTAGE_ADJUSTMENT,POSTAGE_ADJUSTMENT,3511
7,SALE,DELIVERY_CHARGE,267
8,SALE,MARKETPLACE_FEE,3
9,SALE,SALE,1014246


## 5. Star schema and referential integrity audit

Expected objects: `analytics.dim_customer`, `analytics.dim_product`, and `analytics.fact_sales`. The fact grain is one row per staged source transaction line, with unique `staging_id`. The report records one unknown member (`key = 0`) in each dimension and foreign keys from fact to customer, product, and staging.

In [5]:
if engine is not None:
    try:
        with engine.connect() as conn:
            counts = pd.read_sql_query(text("""
                SELECT 'analytics.dim_customer' AS object_name, COUNT(*) AS row_count FROM analytics.dim_customer
                UNION ALL SELECT 'analytics.dim_product', COUNT(*) FROM analytics.dim_product
                UNION ALL SELECT 'analytics.fact_sales', COUNT(*) FROM analytics.fact_sales
            """), conn)
            fact = pd.read_sql_query(text("""
                SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT staging_id) AS distinct_staging_ids
                FROM analytics.fact_sales
            """), conn)
            unknown = pd.read_sql_query(text("""
                SELECT (SELECT COUNT(*) FROM analytics.dim_customer WHERE customer_key=0) AS unknown_customers,
                       (SELECT COUNT(*) FROM analytics.dim_product WHERE product_key=0) AS unknown_products
            """), conn)
            orphans = pd.read_sql_query(text("""
                SELECT COUNT(*) FILTER (WHERE c.customer_key IS NULL) AS orphan_customer_rows,
                       COUNT(*) FILTER (WHERE p.product_key IS NULL) AS orphan_product_rows,
                       COUNT(*) FILTER (WHERE s.staging_id IS NULL) AS orphan_staging_rows
                FROM analytics.fact_sales f
                LEFT JOIN analytics.dim_customer c ON f.customer_key=c.customer_key
                LEFT JOIN analytics.dim_product p ON f.product_key=p.product_key
                LEFT JOIN staging.retail_transactions s ON f.staging_id=s.staging_id
            """), conn)
        display(counts); display(fact); display(unknown); display(orphans)
        count_map = dict(zip(counts.object_name,counts.row_count))
        compare("dim_customer row count",int(count_map["analytics.dim_customer"]),BASELINE["customers"])
        compare("dim_product row count",int(count_map["analytics.dim_product"]),BASELINE["products"])
        compare("fact_sales row count",int(count_map["analytics.fact_sales"]),BASELINE["fact"])
        compare("Fact staging_id uniqueness",int(fact.distinct_staging_ids.iloc[0]),int(fact.fact_rows.iloc[0]))
        compare("Unknown customer member count",int(unknown.unknown_customers.iloc[0]),1)
        compare("Unknown product member count",int(unknown.unknown_products.iloc[0]),1)
        for col in orphans.columns:
            compare(f"Zero {col}",int(orphans[col].iloc[0]),0)
    except Exception as exc:
        print("Star schema audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Star schema audit","NOT VERIFIED",type(exc).__name__,"Dimensions, fact, keys and orphan checks")

,object_name,row_count
0,analytics.dim_customer,5943
1,analytics.dim_product,5305
2,analytics.fact_sales,1044848


,fact_rows,distinct_staging_ids
0,1044848,1044848


,unknown_customers,unknown_products
0,1,1


,orphan_customer_rows,orphan_product_rows,orphan_staging_rows
0,0,0,0


### 5.1 Declared PostgreSQL foreign-key verification

This check verifies the three declared foreign-key constraints, not only the absence of orphan rows. It compares the child and parent schemas, tables, and columns against the implemented star-schema relationships.

In [6]:
if engine is not None:
    expected_fks = {
        ("analytics", "fact_sales", "customer_key", "analytics", "dim_customer", "customer_key"),
        ("analytics", "fact_sales", "product_key", "analytics", "dim_product", "product_key"),
        ("analytics", "fact_sales", "staging_id", "staging", "retail_transactions", "staging_id"),
    }
    try:
        with engine.connect() as conn:
            declared_fks = pd.read_sql_query(text("""
                SELECT
                    tc.constraint_name,
                    tc.constraint_schema AS child_schema,
                    kcu.table_name AS child_table,
                    kcu.column_name AS child_column,
                    ccu.table_schema AS parent_schema,
                    ccu.table_name AS parent_table,
                    ccu.column_name AS parent_column
                FROM information_schema.table_constraints tc
                JOIN information_schema.key_column_usage kcu
                  ON tc.constraint_name = kcu.constraint_name
                 AND tc.constraint_schema = kcu.constraint_schema
                 AND tc.table_name = kcu.table_name
                JOIN information_schema.constraint_column_usage ccu
                  ON tc.constraint_name = ccu.constraint_name
                 AND tc.constraint_schema = ccu.constraint_schema
                WHERE tc.constraint_type = 'FOREIGN KEY'
                  AND tc.constraint_schema = 'analytics'
                  AND tc.table_name = 'fact_sales'
                ORDER BY kcu.column_name
            """), conn)
        display(declared_fks)
        observed_fks = set(
            declared_fks[[
                "child_schema", "child_table", "child_column",
                "parent_schema", "parent_table", "parent_column"
            ]].itertuples(index=False, name=None)
        )
        missing_fks = sorted(expected_fks - observed_fks)
        unexpected_fks = sorted(observed_fks - expected_fks)
        fk_status = "PASS" if not missing_fks and not unexpected_fks else "FAIL"
        add_check("Declared fact_sales foreign keys", fk_status,
                  sorted(observed_fks), sorted(expected_fks),
                  f"Missing: {missing_fks}; unexpected: {unexpected_fks}")
    except Exception as exc:
        print("Foreign-key constraint audit unavailable:", type(exc).__name__, str(exc)[:250])
        add_check("Declared fact_sales foreign keys", "NOT VERIFIED",
                  type(exc).__name__, sorted(expected_fks))


,constraint_name,child_schema,child_table,child_column,parent_schema,parent_table,parent_column
0,fact_sales_customer_key_fkey,analytics,fact_sales,customer_key,analytics,dim_customer,customer_key
1,fact_sales_product_key_fkey,analytics,fact_sales,product_key,analytics,dim_product,product_key
2,fact_sales_staging_id_fkey,analytics,fact_sales,staging_id,staging,retail_transactions,staging_id


## 6. Single Customer View audit

The view should contain one row per identified customer (unknown member excluded). The report records 5,942 customers, 5,852 merchandise purchasers, 90 customers without merchandise orders, snapshot date 2011-12-09, and merchandise revenue £17,117,089.2970. Recency is based on the dataset snapshot, not the system date.

In [7]:
if engine is not None:
    try:
        with engine.connect() as conn:
            scv = pd.read_sql_query(text("""
                SELECT COUNT(*) AS identified_customer_rows,
                       COUNT(DISTINCT customer_key) AS distinct_customer_keys,
                       COUNT(*) FILTER (WHERE total_orders > 0) AS merchandise_purchasers,
                       COUNT(*) FILTER (WHERE total_orders = 0) AS customers_without_merchandise_orders,
                       SUM(total_merchandise_revenue) AS merchandise_revenue,
                       MAX(snapshot_date) AS snapshot_date
                FROM analytics.vw_single_customer
            """), conn)
        display(scv)
        r = scv.iloc[0]
        compare("SCV identified customers",int(r.identified_customer_rows),BASELINE["scv_customers"])
        compare("SCV distinct customer keys",int(r.distinct_customer_keys),int(r.identified_customer_rows))
        compare("SCV merchandise purchasers",int(r.merchandise_purchasers),BASELINE["purchasers"])
        compare("SCV customers without orders",int(r.customers_without_merchandise_orders),BASELINE["no_orders"])
        compare("SCV merchandise revenue",float(r.merchandise_revenue),BASELINE["scv_revenue"],tol=0.01)
        compare("SCV snapshot date",str(pd.to_datetime(r.snapshot_date).date()),"2011-12-09")
    except Exception as exc:
        print("SCV audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Single Customer View audit","NOT VERIFIED",type(exc).__name__,"analytics.vw_single_customer")

,identified_customer_rows,distinct_customer_keys,merchandise_purchasers,customers_without_merchandise_orders,merchandise_revenue,snapshot_date
0,5942,5942,5852,90,1.711709e+07,2011-12-09


## 7. Repository and SQL asset audit

File presence is evidence that an asset exists, not proof that its migration executed. Runtime database checks above and below provide separate execution evidence.

The current implementation includes migration 006 for the materialized customer metrics and query 007 for explicit metric refresh and validation.


In [8]:
assets = [
    MIGRATIONS/"001_create_staging_table.sql",
    MIGRATIONS/"002_create_cleaned_transactions.sql",
    MIGRATIONS/"003_create_cleaned_transactions_v2.sql",
    MIGRATIONS/"004_create_star_schema.sql",
    MIGRATIONS/"005_create_single_customer_view.sql",
    MIGRATIONS/"006_optimize_single_customer_view.sql",
    ROOT/"database"/"queries"/"007_refresh_customer_metrics.sql",
    ERD, BENCHMARK, REPORT
]
asset_df = pd.DataFrame([{"asset":str(p.relative_to(ROOT)),"exists":p.exists(),
                          "size_bytes":p.stat().st_size if p.exists() else None} for p in assets])
display(asset_df)
for _, a in asset_df.iterrows():
    add_check(f"Asset exists: {a.asset}","PASS" if a.exists else "FAIL",bool(a.exists),True)

# The refresh is an explicit operational step, not an automatic staging-loader action.
refresh_sql = ROOT/"database"/"queries"/"007_refresh_customer_metrics.sql"
if refresh_sql.exists():
    refresh_text = refresh_sql.read_text(encoding="utf-8")
    add_check("Refresh SQL contains materialized-view refresh",
              "PASS" if "REFRESH MATERIALIZED VIEW analytics.mv_customer_metrics" in refresh_text
              else "REVIEW",
              "Refresh statement found" if "REFRESH MATERIALIZED VIEW analytics.mv_customer_metrics" in refresh_text
              else "Expected refresh statement not found",
              "REFRESH MATERIALIZED VIEW analytics.mv_customer_metrics")


,asset,exists,size_bytes
0,database\migrations\001_create_staging_table.sql,True,1485
1,database\migrations\002_create_cleaned_transac...,True,2275
2,database\migrations\003_create_cleaned_transac...,True,2223
3,database\migrations\004_create_star_schema.sql,True,8930
4,database\migrations\005_create_single_customer...,True,3719
5,database\migrations\006_optimize_single_custom...,True,4298
6,database\queries\007_refresh_customer_metrics.sql,True,466
7,database\diagrams\consumer360_star_schema.mmd,True,1097
8,database\queries\006_performance_benchmark.sql,True,1206
9,reports\ingestion_report.json,True,4620


## 8. SQL performance audit

The table below preserves both the earlier historical benchmark and the later optimized measurements recorded after migration 006 was applied to the populated local PostgreSQL database.

The optimized values are recorded evidence from that local benchmark run. They are not claimed to have been freshly re-measured by executing this notebook. A fresh run of the benchmark SQL is a separate machine- and environment-specific measurement.


In [9]:
if BENCHMARK.exists():
    print(BENCHMARK.read_text(encoding="utf-8"))
    add_check("Benchmark SQL readable","PASS",True,True)
else:
    add_check("Benchmark SQL readable","FAIL",False,True)

performance_evidence = pd.DataFrame([
    ("Original baseline — customer lookup (18102)",1045.764,"Historical pre-optimization"),
    ("Original baseline — top 10 customers",1544.473,"Historical pre-optimization"),
    ("Original baseline — country-level revenue",1891.806,"Historical pre-optimization"),
    ("Optimized — customer lookup (18102)",195.495,"Recorded local benchmark after migration 006"),
    ("Optimized — top 10 customers",9.469,"Recorded local benchmark after migration 006"),
    ("Optimized — country-level revenue",9.040,"Recorded local benchmark after migration 006"),
],columns=["query","execution_ms","evidence_provenance"])
performance_evidence["target_ms"] = BASELINE["target_ms"]
performance_evidence["reported_result"] = performance_evidence.execution_ms.map(
    lambda x:"PASS" if x < BASELINE["target_ms"] else "FAIL"
)
display(performance_evidence)

for _, row in performance_evidence.iterrows():
    add_check(f"Recorded performance: {row['query']}",row.reported_result,
              float(row.execution_ms),f"< {BASELINE['target_ms']} ms",
              f"{row.evidence_provenance}; not re-measured in this notebook run.")

# Keep the latest optimized results separately visible for a concise acceptance check.
optimized = performance_evidence[performance_evidence["evidence_provenance"].str.startswith("Recorded local")]
compare("Optimized benchmark query count",len(optimized),3)
compare("Optimized benchmark queries below 2 seconds",
        int((optimized.execution_ms < BASELINE["target_ms"]).sum()),3)


\echo '===================================================='
\echo 'TEST 1: Single customer lookup'
\echo '===================================================='

EXPLAIN (ANALYZE, BUFFERS, SUMMARY)
SELECT
    customer_id,
    total_orders,
    total_merchandise_revenue,
    average_order_value,
    recency_days
FROM analytics.vw_single_customer
WHERE customer_id = '18102';

\echo '===================================================='
\echo 'TEST 2: Top 10 customers by merchandise revenue'
\echo '===================================================='

EXPLAIN (ANALYZE, BUFFERS, SUMMARY)
SELECT
    customer_id,
    country,
    total_orders,
    total_merchandise_revenue
FROM analytics.vw_single_customer
ORDER BY total_merchandise_revenue DESC
LIMIT 10;

\echo '===================================================='
\echo 'TEST 3: Country-level merchandise revenue'
\echo '===================================================='

EXPLAIN (ANALYZE, BUFFERS, SUMMARY)
SELECT
    country,
    COUNT

,query,execution_ms,evidence_provenance,target_ms,reported_result
0,Original baseline — customer lookup (18102),1045.764,Historical pre-optimization,2000.0,PASS
1,Original baseline — top 10 customers,1544.473,Historical pre-optimization,2000.0,PASS
2,Original baseline — country-level revenue,1891.806,Historical pre-optimization,2000.0,PASS
3,Optimized — customer lookup (18102),195.495,Recorded local benchmark after migration 006,2000.0,PASS
4,Optimized — top 10 customers,9.469,Recorded local benchmark after migration 006,2000.0,PASS
5,Optimized — country-level revenue,9.040,Recorded local benchmark after migration 006,2000.0,PASS


## 9. Week 1 requirement traceability

| Week 1 requirement | Evidence reviewed |
|---|---|
| Foundational Star Schema | Dimension/fact counts, fact grain, keys and relationships |
| Clean raw transaction logs | Source reconciliation, staging lineage, quality signals and classification |
| Single Customer View | View grain, customer counts, snapshot and revenue |
| ER Diagram validation | ERD asset, declared PostgreSQL foreign keys, and orphan checks |
| SQL performance under 2 seconds | Historical and optimized recorded `EXPLAIN ANALYZE` timings; fresh run remains separate |
| Optimized customer metrics | Migration 006, materialized-view refresh query 007, and recorded optimized benchmark |


## 10. Exceptions and interpretation notes

The Week 1 report documents these caveats:

- Negative quantity is not automatically a confirmed refund; cancellation and negative-quantity flags were handled separately.
- Candidate line amount is not approved accounting revenue by itself.
- Merchandise and non-merchandise amounts are separated.
- Missing customer IDs remain represented by an unknown customer member; they are not fabricated.
- Single Customer View revenue is scoped to identified customers and may differ from the full fact-table merchandise amount.
- The optimized customer lookup still calculates the snapshot date from the fact table; the materialized metrics optimize customer-level aggregation but do not eliminate every fact-table scan.
- Optimized benchmark values are recorded local measurements and are not guaranteed runtimes on other machines.
- Source file and migration file presence do not prove database execution.
- Streamlit Cloud deployment and remote analytical-data hosting are future implementation tasks; they are not evidence of Week 1 completion.

Add issues discovered during execution below. Do not change source data or business definitions merely to force a pass.


## 11. Consolidated audit verdict

`PASS` means every required Week 1 check executed and passed. `FAIL` means at least one required check failed. `INCOMPLETE` means no confirmed failure exists, but evidence is missing or requires review. `NOT VERIFIED` is never a pass.

The deployment inventory is informational and is not included in the Week 1 verdict.


In [10]:
audit = pd.DataFrame(checks)
display(audit)
if audit.empty:
    verdict = "INCOMPLETE"
else:
    counts = audit.status.value_counts().to_dict()
    if counts.get("FAIL",0):
        verdict = "FAIL"
    elif counts.get("NOT VERIFIED",0) or counts.get("REVIEW",0):
        verdict = "INCOMPLETE"
    else:
        verdict = "PASS"
print("OVERALL WEEK 1 AUDIT VERDICT:",verdict)
print("Check counts:",audit.status.value_counts().to_dict())

,check,status,observed,expected,note
0,Required columns: online_retail_II_2009-10.csv,PASS,[],[],NaN
1,Required columns: online_retail_II_2010-11.csv,PASS,[],[],NaN
2,2009–10 source row count,PASS,525461,525461,NaN
3,2010–11 source row count,PASS,541910,541910,NaN
4,Cross-file overlap occurrences,PASS,22523,22523,NaN
5,Retained rows after overlap handling,PASS,1044848,1044848,NaN
6,PostgreSQL connection,PASS,Connected,Connected,NaN
7,Staging row count,PASS,1044848,1044848,NaN
8,Staging ID uniqueness,PASS,1044848,1044848,NaN
9,Staging lineage completeness,PASS,0,0,NaN


OVERALL WEEK 1 AUDIT VERDICT: PASS
Check counts: {'PASS': 47}


## 12. Week 1 audit conclusion

This notebook audits only the Week 1 data foundation. The verdict above is calculated from the checks that could be executed in the current environment.

- `PASS`: all available required checks passed.
- `INCOMPLETE`: required evidence was unavailable or needs review.
- `FAIL`: at least one required control failed.

No manual sign-off fields or exception entries are required. If a database or source file is unavailable, the notebook records that check as `NOT VERIFIED` rather than stopping execution. A `NOT VERIFIED` result is not a pass.
